In [ ]:
# Claude Thread Link
# https://claude.ai/share/a4550489-da25-4570-a840-3248dc324833

In [1]:
!pip install stable-baselines3

In [2]:
!pip install shimmy

In [15]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

In [4]:
# Create directories for logs and models
log_dir = "logs/"
model_dir = "models/"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

In [5]:
def load_data(data_path):
    try:
        data = pd.read_excel(data_path)
        print(f"Successfully loaded data from {data_path}")
        print(f"Data shape: {data.shape}")
        print(f"Column names: {data.columns.tolist()}")

        # Display sample of the data
        display(data.head())

        return data
    except Exception as e:
        print(f"Failed to load data: {e}")
        return None

In [6]:
data_path = "/content/drive/MyDrive/Capstone Project/TSX Chart Data.xlsx"
custom_data = load_data(data_path)

Successfully loaded data from /content/drive/MyDrive/Capstone Project/TSX Chart Data.xlsx
Data shape: (1048575, 8)
Column names: ['date', 'close', 'high', 'low', 'open', 'volume', 'tic', 'day']


,date,close,high,low,open,volume,tic,day
0,2004-01-02,0.489625,0.53,0.53,0.53,0,AAB.TO,4
1,2004-01-05,0.489625,0.53,0.53,0.53,0,AAB.TO,0
2,2004-01-06,0.258670,0.28,0.28,0.28,100,AAB.TO,1
3,2004-01-07,0.489625,0.53,0.53,0.53,0,AAB.TO,2
4,2004-01-08,0.489625,0.53,0.53,0.53,0,AAB.TO,3


In [7]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

def prepare_stock_data(file_path):
    """
    Prepare stock data from Excel/CSV file for risk analysis

    Args:
        file_path (str): Path to the Excel/CSV file

    Returns:
        dict: Dictionary of DataFrames with processed data for each stock
    """
    # Determine file type and read accordingly
    if file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        df = pd.read_excel(file_path)
    else:  # Assume CSV
        df = pd.read_csv(file_path)

    print(f"Original data shape: {df.shape}")

    # Convert date to datetime if not already
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'])

    # Sort by ticker and date
    df = df.sort_values(['tic', 'date'])

    # Calculate additional features for risk assessment
    stock_dfs = {}
    skipped_stocks = 0

    # Process each stock separately
    for ticker in df['tic'].unique():
        stock_data = df[df['tic'] == ticker].copy()

        # Ensure we have enough data (at least 1 year)
        if len(stock_data) < 252:  # Approx trading days in a year
            print(f"Skipping {ticker} - insufficient data ({len(stock_data)} rows)")
            skipped_stocks += 1
            continue

        # Calculate daily returns
        stock_data['daily_return'] = stock_data['close'].pct_change()

        # Calculate volatility (20-day and 60-day)
        stock_data['volatility_20d'] = stock_data['daily_return'].rolling(window=20).std() * np.sqrt(252)
        stock_data['volatility_60d'] = stock_data['daily_return'].rolling(window=60).std() * np.sqrt(252)

        # Calculate drawdowns
        stock_data['cumulative_return'] = (1 + stock_data['daily_return']).cumprod()
        stock_data['rolling_max'] = stock_data['cumulative_return'].cummax()
        stock_data['drawdown'] = (stock_data['cumulative_return'] / stock_data['rolling_max']) - 1
        stock_data['max_drawdown_60d'] = stock_data['drawdown'].rolling(window=60).min()

        # Calculate trading volume metrics
        stock_data['volume_change'] = stock_data['volume'].pct_change()
        stock_data['volume_ma10'] = stock_data['volume'].rolling(window=10).mean()
        stock_data['relative_volume'] = stock_data['volume'] / stock_data['volume_ma10']

        # Drop NaN values that result from calculations
        stock_data = stock_data.dropna()

        # Only include stock if we still have meaningful data after calculations
        if len(stock_data) >= 126:  # At least ~6 months of data
            stock_dfs[ticker] = stock_data
        else:
            skipped_stocks += 1

    print(f"Processed {len(stock_dfs)} stocks, skipped {skipped_stocks} stocks")
    return stock_dfs

def calculate_risk_metrics(stock_dfs, lookback_period=252):
    """
    Calculate risk metrics for each stock

    Args:
        stock_dfs (dict): Dictionary of stock DataFrames
        lookback_period (int): Period to use for calculations (default 1 year)

    Returns:
        DataFrame: Summary of risk metrics for all stocks
    """
    risk_metrics = []

    for ticker, data in stock_dfs.items():
        # Use recent data only
        recent_data = data.iloc[-lookback_period:] if len(data) > lookback_period else data

        # Skip stocks with zero or constant prices (no price movement)
        if recent_data['daily_return'].std() == 0 or recent_data['close'].nunique() <= 1:
            print(f"Skipping {ticker} - no price variation")
            continue

        # Calculate key risk metrics
        metrics = {
            'ticker': ticker,
            'avg_daily_return': recent_data['daily_return'].mean(),
            'annualized_return': recent_data['daily_return'].mean() * 252,
            'volatility': recent_data['daily_return'].std() * np.sqrt(252),
            'max_drawdown': recent_data['drawdown'].min(),
            'sharpe_ratio': (recent_data['daily_return'].mean() * 252) /
                           (recent_data['daily_return'].std() * np.sqrt(252))
                           if recent_data['daily_return'].std() > 0 else 0,
            'avg_volume': recent_data['volume'].mean(),
            'volume_volatility': recent_data['volume_change'].std() if not pd.isna(recent_data['volume_change'].std()) else 0,
            'data_points': len(recent_data)
        }

        risk_metrics.append(metrics)

    # Convert to DataFrame
    if not risk_metrics:
        print("No valid stocks with risk metrics found!")
        return pd.DataFrame()

    risk_df = pd.DataFrame(risk_metrics)

    # Handle extreme values and missing data
    for col in ['volatility', 'max_drawdown', 'volume_volatility', 'sharpe_ratio']:
        # Replace infinity values
        risk_df[col] = risk_df[col].replace([np.inf, -np.inf], np.nan)

        # Handle missing values with median imputation
        if risk_df[col].isna().any():
            median_val = risk_df[col].median()
            risk_df[col] = risk_df[col].fillna(median_val)

    # Calculate risk score components
    vol_component = (risk_df['volatility'] / risk_df['volatility'].max()) if risk_df['volatility'].max() > 0 else 0
    dd_component = (risk_df['max_drawdown'].abs() / risk_df['max_drawdown'].abs().max()) if risk_df['max_drawdown'].abs().max() > 0 else 0
    vol_vol_component = (risk_df['volume_volatility'] / risk_df['volume_volatility'].max()) if risk_df['volume_volatility'].max() > 0 else 0

    # Sharpe ratio is inversely related to risk (higher is better)
    sharpe_range = risk_df['sharpe_ratio'].max() - risk_df['sharpe_ratio'].min()
    if sharpe_range > 0:
        sharpe_component = 1 - ((risk_df['sharpe_ratio'] - risk_df['sharpe_ratio'].min()) / sharpe_range)
    else:
        sharpe_component = 0.5  # Neutral value if all stocks have the same Sharpe ratio

    # Add weighted risk score calculation
    risk_df['risk_score'] = (
        vol_component * 0.4 +
        dd_component * 0.3 +
        vol_vol_component * 0.15 +
        sharpe_component * 0.15
    )

    # Ensure all stocks have a valid risk score
    if risk_df['risk_score'].isna().any():
        print(f"Warning: {risk_df['risk_score'].isna().sum()} stocks have NaN risk scores")
        # Assign median risk score to stocks with NaN
        risk_df['risk_score'] = risk_df['risk_score'].fillna(risk_df['risk_score'].median())

    # Ensure we don't have exactly 0 or 1 risk scores (adjust slightly if needed)
    risk_df.loc[risk_df['risk_score'] == 0, 'risk_score'] = 0.001
    risk_df.loc[risk_df['risk_score'] == 1, 'risk_score'] = 0.999

    # Classify risk levels using qcut for equal-sized groups
    try:
        risk_df['risk_category'] = pd.qcut(
            risk_df['risk_score'],
            q=3,
            labels=['Low', 'Medium', 'High']
        )
    except ValueError:
        # Fall back to manual classification if qcut fails
        risk_thresholds = [0, 0.33, 0.67, 1.0]
        risk_df['risk_category'] = pd.cut(
            risk_df['risk_score'],
            bins=risk_thresholds,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

    return risk_df

def save_risk_analysis(risk_df, output_path='stock_risk_analysis.csv'):
    """
    Save risk analysis results to CSV file

    Args:
        risk_df (DataFrame): Risk metrics for all stocks
        output_path (str): Path to save the CSV file
    """
    if risk_df.empty:
        print("No data to save!")
        return risk_df

    # Select relevant columns for output
    output_columns = [
        'ticker', 'risk_category', 'risk_score', 'volatility',
        'max_drawdown', 'sharpe_ratio', 'avg_daily_return', 'annualized_return',
        'avg_volume', 'volume_volatility', 'data_points'
    ]

    # Ensure all expected columns exist
    for col in output_columns:
        if col not in risk_df.columns:
            print(f"Warning: Column {col} missing from results")

    available_columns = [col for col in output_columns if col in risk_df.columns]

    # Sort by risk score
    risk_df_sorted = risk_df.sort_values('risk_score')[available_columns]

    # Save to CSV
    risk_df_sorted.to_csv(output_path, index=False)
    print(f"Risk analysis saved to {output_path}")

    return risk_df_sorted

# Main execution
if __name__ == "__main__":
    file_path = "/content/drive/MyDrive/Capstone Project/TSX Chart Data.xlsx"

    # Process data
    stock_dfs = prepare_stock_data(file_path)

    # Calculate risk metrics
    risk_df = calculate_risk_metrics(stock_dfs)

    if not risk_df.empty:
        # Save results
        final_df = save_risk_analysis(risk_df)

        print("\nRisk Distribution:")
        print(risk_df['risk_category'].value_counts())

        print("\nSample of High Risk Stocks:")
        print(risk_df[risk_df['risk_category'] == 'High'].head(5)[['ticker', 'risk_score', 'volatility', 'max_drawdown']])

        print("\nSample of Low Risk Stocks:")
        print(risk_df[risk_df['risk_category'] == 'Low'].head(5)[['ticker', 'risk_score', 'volatility', 'max_drawdown']])
    else:
        print("No valid risk data generated!")

Original data shape: (1048575, 8)
Skipping ACAA.TO - insufficient data (120 rows)
Skipping ADIV.TO - insufficient data (183 rows)
Skipping AGG.TO - insufficient data (50 rows)
Skipping AIGO.TO - insufficient data (155 rows)
Skipping AMAX.TO - insufficient data (224 rows)
Skipping AMHE.TO - insufficient data (88 rows)
Skipping AMZH.TO - insufficient data (89 rows)
Skipping AW.TO - insufficient data (49 rows)
Skipping BCHT.TO - insufficient data (46 rows)
Skipping CAPG.TO - insufficient data (45 rows)
Skipping CAPI.TO - insufficient data (45 rows)
Skipping CAPM.TO - insufficient data (45 rows)
Skipping CAPW.TO - insufficient data (45 rows)
Skipping CBL.TO - insufficient data (139 rows)
Skipping CIAI.TO - insufficient data (162 rows)
Skipping CMOM.TO - insufficient data (240 rows)
Skipping CNDX.TO - insufficient data (155 rows)
Skipping CNV.TO - insufficient data (139 rows)
Skipping COPR.TO - insufficient data (95 rows)
Skipping CORE.TO - insufficient data (91 rows)
Skipping CSBI.TO - ins

In [16]:
# Create a custom gym environment for stock risk assessment
class StockRiskEnv(gym.Env):
    """Custom Environment for stock risk assessment"""
    metadata = {'render.modes': ['human']}

    def __init__(self, stock_data_dict, feature_columns):
        super(StockRiskEnv, self).__init__()

        # Store stock data
        self.stock_data_dict = stock_data_dict
        self.feature_columns = feature_columns
        self.tickers = list(stock_data_dict.keys())

        # Define action and observation space
        # Actions: 0 = Low Risk, 1 = Medium Risk, 2 = High Risk
        self.action_space = spaces.Discrete(3)

        # Use a simpler observation space with fixed bounds
        self.num_features = len(feature_columns) * 5  # 5 metrics per feature
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(self.num_features,), dtype=np.float32
        )

        # Environment variables
        self.current_ticker = None
        self.current_step = 0
        self.max_steps = 20
        self.window_size = 30

        # For handling NaN and Inf values
        self.valid_tickers = self._get_valid_tickers()
        if not self.valid_tickers:
            raise ValueError("No valid tickers found with sufficient data")

    def _get_valid_tickers(self):
        """Filter tickers with valid data"""
        valid_tickers = []
        for ticker, data in self.stock_data_dict.items():
            # Check if ticker has enough data
            if len(data) >= self.window_size + self.max_steps:
                # Check if features exist and don't have too many NaNs
                feature_exists = all(col in data.columns for col in self.feature_columns)
                if feature_exists:
                    # Count NaN values in relevant features
                    nan_count = data[self.feature_columns].isna().sum().sum()
                    if nan_count / (len(data) * len(self.feature_columns)) < 0.1:  # Less than 10% NaNs
                        valid_tickers.append(ticker)

        print(f"Found {len(valid_tickers)} valid tickers out of {len(self.stock_data_dict)}")
        return valid_tickers

    def reset(self, seed=None, options=None):
        """Reset the environment to start a new episode"""
        super().reset(seed=seed)

        # Choose a random ticker from valid tickers
        if not self.valid_tickers:
            raise ValueError("No valid tickers available")

        self.current_ticker = np.random.choice(self.valid_tickers)
        self.stock_data = self.stock_data_dict[self.current_ticker].copy()

        # Fill NaN values to prevent training issues
        for col in self.feature_columns:
            if col in self.stock_data.columns:
                self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)

        # Start at a random position with enough history
        max_start = len(self.stock_data) - self.window_size - self.max_steps
        if max_start <= 0:
            # If not enough data, try another ticker
            return self.reset(seed=seed)

        self.current_position = np.random.randint(self.window_size, max_start)
        self.current_step = 0

        # Get initial state
        observation = self._get_observation()

        # Additional check for NaN values
        if np.isnan(observation).any() or np.isinf(observation).any():
            # Return zeros instead of NaN
            observation = np.zeros(self.observation_space.shape, dtype=np.float32)

        return observation, {}

    def step(self, action):
        """Take a step in the environment"""
        self.current_step += 1
        done = self.current_step >= self.max_steps

        # Calculate reward based on risk levels
        try:
            # Get current state data
            current_data = self.stock_data.iloc[self.current_position - self.window_size:self.current_position]

            # Calculate true risk based on future performance (next 30 days)
            future_window = min(30, len(self.stock_data) - self.current_position)
            future_data = self.stock_data.iloc[self.current_position:self.current_position + future_window]

            # Calculate risk metrics with more nuanced thresholds
            future_return = future_data['daily_return'].mean() * 252 if 'daily_return' in future_data.columns else 0
            future_volatility = np.std(future_data['daily_return']) * np.sqrt(252) if 'daily_return' in future_data.columns else 0
            future_max_drawdown = future_data['drawdown'].min() if 'drawdown' in future_data.columns else 0

            # Calculate Sharpe ratio (risk-adjusted return)
            risk_free_rate = 0.02  # Assume 2% risk-free rate
            sharpe_ratio = (future_return - risk_free_rate) / future_volatility if future_volatility > 0 else 0

            # More nuanced risk level classification with clearer medium category
            # Use multiple factors to determine risk level
            risk_score = 0
            risk_score += 1 if future_volatility > 0.3 else (0.5 if future_volatility > 0.15 else 0)
            risk_score += 1 if future_max_drawdown < -0.15 else (0.5 if future_max_drawdown < -0.08 else 0)
            risk_score += 0.5 if sharpe_ratio < 0 else (0 if sharpe_ratio > 1 else 0.25)

            # Map risk score to risk level
            if risk_score > 1.5:
                true_risk = 2  # High risk
            elif risk_score > 0.75:
                true_risk = 1  # Medium risk
            else:
                true_risk = 0  # Low risk

            # Enhanced reward function with explicit incentive for medium risk
            if action == true_risk:
                reward = 1.5  # Reward for correct classification
                if true_risk == 1:  # Extra reward for correctly identifying medium risk
                    reward += 1.0
            else:
                # Penalty based on how far off the prediction is
                reward = -0.5 - abs(action - true_risk) * 0.5

        except Exception as e:
            # Fallback if calculation fails
            reward = 0
            true_risk = 0

        # Move to next position
        self.current_position += 1

        # Get next observation
        next_obs = self._get_observation()

        # Safety check for NaN/Inf values
        if np.isnan(next_obs).any() or np.isinf(next_obs).any():
            # Replace problematic values with zeros
            next_obs = np.zeros(self.observation_space.shape, dtype=np.float32)

        info = {
            'ticker': self.current_ticker,
            'true_risk': true_risk
        }

        return next_obs, reward, done, False, info

    def _get_observation(self):
        """Get the current observation (state)"""
        # Get window of data
        window_data = self.stock_data.iloc[self.current_position - self.window_size:self.current_position]

        # Extract features
        features = []
        for col in self.feature_columns:
            if col in window_data.columns:
                # Calculate summary statistics
                values = window_data[col].values

                # Replace NaN and inf values
                values = np.nan_to_num(values, nan=0.0, posinf=10.0, neginf=-10.0)

                if len(values) > 0:
                    feature_mean = np.mean(values)
                    feature_std = np.std(values)
                    feature_min = np.min(values)
                    feature_max = np.max(values)
                    feature_last = values[-1]
                else:
                    feature_mean = feature_std = feature_min = feature_max = feature_last = 0.0

                # Clip extreme values
                feature_mean = np.clip(feature_mean, -10.0, 10.0)
                feature_std = np.clip(feature_std, 0.0, 10.0)
                feature_min = np.clip(feature_min, -10.0, 10.0)
                feature_max = np.clip(feature_max, -10.0, 10.0)
                feature_last = np.clip(feature_last, -10.0, 10.0)

                features.extend([feature_mean, feature_std, feature_min, feature_max, feature_last])
            else:
                # Pad with zeros if feature not available
                features.extend([0.0, 0.0, 0.0, 0.0, 0.0])

        # Final check for NaN/Inf
        features = np.array(features, dtype=np.float32)
        features = np.nan_to_num(features, nan=0.0, posinf=10.0, neginf=-10.0)

        return features

    def render(self):
        """Render the environment"""
        print(f"Ticker: {self.current_ticker}, Step: {self.current_step}")

def train_a2c_model(stock_dfs, total_timesteps=100000):
    """
    Train an A2C model using Stable Baselines 3

    Args:
        stock_dfs: Dictionary of processed stock DataFrames
        total_timesteps: Total training timesteps

    Returns:
        Trained A2C model
    """
    # Define feature columns from available columns
    sample_df = next(iter(stock_dfs.values()))
    available_cols = sample_df.columns.tolist()

    # Select features that exist in the data
    required_features = ['daily_return', 'volatility_20d', 'drawdown', 'volume_change']
    feature_columns = [col for col in required_features if col in available_cols]

    if not feature_columns:
        raise ValueError("No required features found in the data. Make sure you have calculated these features.")

    print(f"Using features: {feature_columns}")

    # Create and wrap the environment
    def make_env():
        try:
            env = StockRiskEnv(stock_dfs, feature_columns)
            env = Monitor(env)
            return env
        except Exception as e:
            print(f"Error creating environment: {str(e)}")
            raise

    # Create environment with error handling
    try:
        env = DummyVecEnv([make_env])

        # Test the environment
        obs = env.reset()
        test_action = env.action_space.sample()
        next_obs, reward, done, info = env.step([test_action])

        print("Environment test successful")

        # Create A2C model with better hyperparameters
        model = A2C(
            "MlpPolicy",
            env,
            learning_rate=0.0005,
            n_steps=8,
            gamma=0.95,
            verbose=1,
            policy_kwargs=dict(
                net_arch=[dict(pi=[128, 64], vf=[128, 64])],  # Deeper network
                activation_fn=torch.nn.Tanh  # Try Tanh instead of ReLU
            ),
            tensorboard_log="./a2c_stock_risk_tensorboard/"
        )

        # Train the model without progress bar
        print(f"Starting A2C training for {total_timesteps} timesteps...")

        # Track progress manually in smaller chunks for stability
        log_interval = total_timesteps // 20
        for i in range(20):
            model.learn(total_timesteps=log_interval, reset_num_timesteps=False, progress_bar=False)
            print(f"Completed {(i+1)*log_interval}/{total_timesteps} timesteps ({(i+1)*5}%)")

        print("A2C training completed")
        return model, env, feature_columns

    except Exception as e:
        print(f"Error during A2C setup/training: {str(e)}")
        raise

def apply_a2c_risk_classification(model, stock_dfs, risk_df, feature_columns):
    """Apply trained A2C model to classify stock risks"""
    # Create a copy of the risk DataFrame
    updated_risk_df = risk_df.copy()

    # Add columns for A2C risk classification
    updated_risk_df['a2c_risk_score'] = np.nan
    updated_risk_df['a2c_risk_category'] = ""

    # Risk category labels
    risk_categories = ['Low', 'Medium', 'High']

    # Create a simple StockRiskEnv with all stocks
    env = StockRiskEnv(stock_dfs, feature_columns)

    # Process each stock
    processed_count = 0
    for ticker in updated_risk_df['ticker'].unique():
        if ticker not in stock_dfs or ticker not in env.valid_tickers:
            continue

        # Get stock data
        stock_data = stock_dfs[ticker].copy()

        # Skip if not enough data
        if len(stock_data) < env.window_size + 10:
            continue

        try:
            # Create a test environment for just this stock
            test_env = StockRiskEnv({ticker: stock_data}, feature_columns)

            # Reset environment
            obs, _ = test_env.reset()

            # Get more exploration with multiple samples per stock
            actions = []
            for _ in range(5):
                action, _ = model.predict(obs, deterministic=False)  # Use stochastic predictions
                actions.append(action)

            # Take most common action
            action = max(set(actions), key=actions.count)

            # Map action to risk category
            a2c_risk_category = risk_categories[action]

            # Calculate a more nuanced risk score
            action_counts = np.zeros(3)
            for a in actions:
                action_counts[a] += 1

            # Weighted risk score between 0-1
            a2c_risk_score = (action_counts[0] * 0.1 + action_counts[1] * 0.5 + action_counts[2] * 0.9) / 5

            # Update the DataFrame
            idx = updated_risk_df.index[updated_risk_df['ticker'] == ticker]
            updated_risk_df.loc[idx, 'a2c_risk_score'] = a2c_risk_score
            updated_risk_df.loc[idx, 'a2c_risk_category'] = a2c_risk_category

            processed_count += 1
            if processed_count % 10 == 0:
                print(f"Processed {processed_count} stocks")

        except Exception as e:
            print(f"Error classifying {ticker}: {str(e)}")

    print(f"Total stocks processed: {processed_count}")
    return updated_risk_df

def compare_risk_methods(risk_df):
    """Compare traditional and A2C risk classifications"""
    print("Starting comparison...")

    if 'a2c_risk_category' not in risk_df.columns:
        print("A2C risk categories not available")
        return None

    # Drop rows with missing values
    risk_df_clean = risk_df.dropna(subset=['risk_category', 'a2c_risk_category'])

    if len(risk_df_clean) == 0:
        print("No data available for comparison after removing NaNs")
        return None

    # Create a comparison table - explicitly create it without using pd.crosstab
    risk_counts = {}
    for trad, a2c in zip(risk_df_clean['risk_category'], risk_df_clean['a2c_risk_category']):
        if trad not in risk_counts:
            risk_counts[trad] = {'Low': 0, 'Medium': 0, 'High': 0}
        risk_counts[trad][a2c] += 1

    # Convert to DataFrame manually
    rows = []
    for trad in ['Low', 'Medium', 'High']:
        if trad in risk_counts:
            row = [risk_counts[trad]['Low'], risk_counts[trad]['Medium'], risk_counts[trad]['High']]
        else:
            row = [0, 0, 0]
        rows.append(row)

    comparison = pd.DataFrame(rows, index=['Low', 'Medium', 'High'],
                             columns=['Low', 'Medium', 'High'])
    comparison.index.name = 'Traditional Method'
    comparison.columns.name = 'A2C Model'

    print("\nRisk Classification Comparison:")
    print(comparison)

    # Calculate agreement percentage manually
    agreement_count = 0
    total_count = len(risk_df_clean)
    for i in range(total_count):
        if risk_df_clean.iloc[i]['risk_category'] == risk_df_clean.iloc[i]['a2c_risk_category']:
            agreement_count += 1

    agreement_pct = (agreement_count / total_count * 100) if total_count > 0 else 0
    print(f"\nAgreement between methods: {agreement_pct:.2f}%")

    # Skip plotting to avoid potential recursion issues
    print("Skipping plot generation to avoid potential recursion issues")

    return comparison

def save_final_risk_analysis(risk_df, output_path='a2c_stock_risk_analysis.csv'):
    """Save the final risk analysis with A2C classifications"""
    # Sort by A2C risk score
    if 'a2c_risk_score' in risk_df.columns and not risk_df['a2c_risk_score'].isna().all():
        risk_df_sorted = risk_df.sort_values('a2c_risk_score')
    else:
        risk_df_sorted = risk_df.sort_values('risk_score')

    # Save to CSV
    risk_df_sorted.to_csv(output_path, index=False)
    print(f"Final risk analysis saved to {output_path}")

    return risk_df_sorted

In [17]:
# Set a lower recursion limit to help identify the issue
current_recursion_limit = sys.getrecursionlimit()
print(f"Current recursion limit: {current_recursion_limit}")
# We're not changing it, just reporting it

# Try with a very small number of timesteps first to test
try:
    # Use a smaller number of timesteps for testing
    model, env, feature_columns = train_a2c_model(stock_dfs, total_timesteps=50000)

    # If successful, save model
    model.save("a2c_stock_risk_model")

    # Apply model to risk classification
    updated_risk_df = apply_a2c_risk_classification(model, stock_dfs, risk_df, feature_columns)

    # Save final results
    final_df = save_final_risk_analysis(updated_risk_df)

    # Compare methods with extra error handling
    try:
        comparison = compare_risk_methods(final_df)
    except Exception as e:
        print(f"Error during comparison: {str(e)}")
        print("Skipping comparison step")

    print("\nA2C Risk Distribution:")
    if 'a2c_risk_category' in final_df.columns:
        print(final_df['a2c_risk_category'].value_counts())
    else:
        print("A2C risk categories not available")

    print("\nTraditional Risk Distribution:")
    print(final_df['risk_category'].value_counts())

except Exception as e:
    print(f"Error in A2C model training: {str(e)}")
    print("Falling back to traditional risk classification only")

Current recursion limit: 1000
Using features: ['daily_return', 'volatility_20d', 'drawdown', 'volume_change']
Found 357 valid tickers out of 357
Environment test successful
Using cpu device
Starting A2C training for 50000 timesteps...
Logging to ./a2c_stock_risk_tensorboard/A2C_4


<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


------------------------------------
| rollout/              |          |
|    ep_len_mean        | 20       |
|    ep_rew_mean        | -1.73    |
| time/                 |          |
|    fps                | 294      |
|    iterations         | 100      |
|    time_elapsed       | 2        |
|    total_timesteps    | 800      |
| train/                |          |
|    entropy_loss       | -0.743   |
|    explained_variance | -0.0127  |
|    learning_rate      | 0.0005   |
|    n_updates          | 99       |
|    policy_loss        | 3.93     |
|    value_loss         | 25.5     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 20       |
|    ep_rew_mean        | 1.27     |
| time/                 |          |
|    fps                | 322      |
|    iterations         | 200      |
|    time_elapsed       | 4        |
|    total_timesteps    | 1600     |
| train/                |          |
|

<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython

Error classifying ALYA.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying AMC.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying AMM.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying AND.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying ANRG.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying AOI.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying AOT.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying APLI.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying APS.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying AQN.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying ARA.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error c

...

Error classifying DIVS.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DLCG.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DLR.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DML.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DND.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DNG.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DNTL.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DOL.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DOO.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DPM.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DR.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error cl

<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython

Error classifying DS.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DSAE.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DSG.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DSV.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DTOL.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DXB.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DXBC.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DXBU.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DXC.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DXDB.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying DXEM.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error

<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython-input-16-09d5c6c9927b>:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.stock_data[col] = self.stock_data[col].fillna(method='ffill').fillna(0)
<ipython

Error classifying E.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying EAGR.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying EARN.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying EBIT.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying EBNK.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying ECN.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying ECO.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying ECOR.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying EDGE.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying EDGF.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error classifying EDR.TO: unhashable type: 'numpy.ndarray'
Found 1 valid tickers out of 1
Error